# Import Libraries

In [ ]:
!pip install cugraph-cu12 --extra-index-url=https://pypi.nvidia.com -q

In [ ]:
!pip install torch_geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 34.8 MB/s eta 0:00:00


In [ ]:
!pip install igraph -q
from igraph import Graph

In [ ]:
import torch
# from torch_geometric.datasets import AMiner, Taobao, MovieLens1M, AmazonBook, HM
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
import time
import psutil
import gc
import sys

In [ ]:
# !git clone https://github.com/yutengkai/Louvain-Algorithm-Without-Graph-Construction.git
!unzip Louvain-Algorithm-Without-Graph-Construction.zip

Archive:  Louvain-Algorithm-Without-Graph-Construction.zip
   creating: Louvain-Algorithm-Without-Graph-Construction/
   creating: Louvain-Algorithm-Without-Graph-Construction/.git/
 extracting: Louvain-Algorithm-Without-Graph-Construction/.git/COMMIT_EDITMSG  
  inflating: Louvain-Algorithm-Without-Graph-Construction/.git/config  
  inflating: Louvain-Algorithm-Without-Graph-Construction/.git/description  
  inflating: Louvain-Algorithm-Without-Graph-Construction/.git/FETCH_HEAD  
 extracting: Louvain-Algorithm-Without-Graph-Construction/.git/HEAD  
   creating: Louvain-Algorithm-Without-Graph-Construction/.git/hooks/
  inflating: Louvain-Algorithm-Without-Graph-Construction/.git/hooks/applypatch-msg.sample  
  inflating: Louvain-Algorithm-Without-Graph-Construction/.git/hooks/commit-msg.sample  
  inflating: Louvain-Algorithm-Without-Graph-Construction/.git/hooks/fsmonitor-watchman.sample  
  inflating: Louvain-Algorithm-Without-Graph-Construction/.git/hooks/post-checkout  
  inflati

In [ ]:
import os
if os.path.exists('Louvain-Algorithm-Without-Graph-Construction') and not os.path.exists('my_louvain'):
    os.rename('Louvain-Algorithm-Without-Graph-Construction', 'my_louvain')

In [ ]:
from my_louvain.src.data.data_downloading import *
from my_louvain.src.data.data_preprocessing import *
from my_louvain.src.main_algorithm import *
from my_louvain.src.utils import *

In [ ]:
import cudf
import cugraph
import cupy as cp

/usr/local/lib/python3.10/dist-packages/cudf/utils/_ptxcompiler.py:64: UserWarning: Error getting driver and runtime versions:

stdout:



stderr:

Traceback (most recent call last):
  File "<string>", line 4, in <module>
  File "/usr/local/lib/python3.10/dist-packages/numba/cuda/cudadrv/driver.py", line 295, in __getattr__
    raise CudaSupportError("Error at driver init: \n%s:" %
numba.cuda.cudadrv.error.CudaSupportError: Error at driver init: 

CUDA driver library cannot be found.
If you are sure that a CUDA driver is installed,
try setting environment variable NUMBA_CUDA_DRIVER
with the file path of the CUDA driver shared library.
:


Not patching Numba
  warnings.warn(msg, UserWarning)
/usr/local/lib/python3.10/dist-packages/cudf/utils/gpu_utils.py:62: UserWarning: Failed to dlopen libcuda.so.1
  warnings.warn(str(e))


ImportError: 
================================================================
Failed to import CuPy.

If you installed CuPy via wheels (cupy-cudaXXX or cupy-rocm-X-X), make sure that the package matches with the version of CUDA or ROCm installed.

On Linux, you may need to set LD_LIBRARY_PATH environment variable depending on how you installed CUDA/ROCm.
On Windows, try setting CUDA_PATH environment variable.

Check the Installation Guide for details:
  https://docs.cupy.dev/en/latest/install.html

Original error:
  ImportError: libcuda.so.1: cannot open shared object file: No such file or directory
================================================================


# Download and Preprocess Data

## Flickr

In [ ]:
from torch_geometric.datasets import Flickr

In [ ]:
Flickr_data = Flickr(root='data/Flickr')

Processing...
Done!


In [ ]:
Flickr_data.data.x.shape

/usr/local/lib/python3.10/dist-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


torch.Size([89250, 500])

In [ ]:
is_row_zero = (Flickr_data.data.x == 0).all(dim=1)
# Flickr_data.data.x[is_row_zero]

In [ ]:
# sub_node_vectors = Flickr_data.data.x[~is_row_zero][:18000, :]
# sub_node_vectors_np = sub_node_vectors.cpu().numpy()  # Convert PyTorch tensor to NumPy
# sub_node_vectors_cp = cp.asarray(sub_node_vectors_np)  # Convert NumPy to CuPy
# resolution=1

In [ ]:
# # Assuming `sub_node_vectors` is a CuPy array or a PyTorch tensor on GPU
# start_time = time.time()

# # Step 1: Compute cosine similarity using CuPy (manually)
# norms = cp.linalg.norm(sub_node_vectors_cp, axis=1)
# cosine_similarity_matrix = cp.dot(sub_node_vectors_cp, sub_node_vectors_cp.T) / (norms[:, None] * norms[None, :])

# # Step 2: Normalize cosine similarity between 0 and 1 (1 - cosine similarity)
# adj = (1 + cosine_similarity_matrix) / 2

# # Step 3: Fill diagonal with zeros (set self-loop weights to 0)
# cp.fill_diagonal(adj, 0)

# # Step 4: Convert adjacency matrix to cuGraph input format (source, destination, weight)
# # Convert adjacency matrix to dataframe of edges
# adj_df = cudf.DataFrame({
#     'src': cp.repeat(cp.arange(adj.shape[0]), adj.shape[1]).get(),
#     'dst': cp.tile(cp.arange(adj.shape[1]), adj.shape[0]).get(),
#     'weight': adj.flatten().get()
# })

# # Remove self-loops (if needed)
# adj_df = adj_df[adj_df['src'] != adj_df['dst']]

# # Step 5: Create cuGraph Graph object and add edges
# G = cugraph.Graph(directed=False)
# G.from_cudf_edgelist(adj_df, source='src', destination='dst', edge_attr='weight')

# # Step 6: Run the Louvain algorithm with a resolution parameter
# louvain_results, modularity_score = cugraph.louvain(G, resolution=resolution)

# # Step 7: Timing
# cu_time = time.time() - start_time

# # Output
# print(f"Modularity: {modularity_score}, Time: {cu_time} seconds")

In [ ]:
# del sub_node_vectors_cp, adj, cosine_similarity_matrix, adj_df, G

# # Explicitly free all memory in the pool
# cp.get_default_memory_pool().free_all_blocks()

In [ ]:
def run_experiments(node_vectors, threshold_list, num_iterations=1, resolution=1):
    """
    Runs experiments for I-graph, cuGraph, and Graph-less Louvain algorithms and records results.

    Parameters:
    - node_vectors (torch.Tensor): Pre-processed tensor (node vectors).
    - threshold_list (list): List of threshold percentages to filter on (0.05, 0.1, etc.).
    - num_iterations (int): Number of times to run the experiment to ensure time stability.

    Returns:
    - pd.DataFrame: DataFrame with experiment results for each threshold.
    """
    results = []

    # Iterate through each threshold
    for threshold in threshold_list:
        row_result = {'threshold': threshold}

        # Step 1: Select top percentage of rows based on threshold
        num_rows = int(node_vectors.size(0) * threshold)
        sub_node_vectors = node_vectors[:num_rows, :]  # Selecting top rows
        row_result['num_nodes'] = num_rows

        # Check if the number of rows exceeds 25,000 for iGraph
        if sub_node_vectors.shape[0] > 18000:
            # Skip I-graph if rows exceed threshold
            row_result['igraph_time'] = 0
            row_result['igraph_modularity'] = 0
        else:
            # Step 2: Run I-graph Partitioning (Cosine Similarity + Louvain)
            igraph_times = []
            igraph_modularities = []
            for _ in range(num_iterations):
                start_time = time.time()

                adj = cosine_similarity(sub_node_vectors.cpu().numpy())  # Cosine similarity for I-graph
                adj = (adj / 2) + 0.5  # Normalize cosine similarity between 0 and 1
                np.fill_diagonal(adj, 0)  # Set diagonal to 0
                ig = Graph.Weighted_Adjacency(adj.tolist(), mode='undirected')  # Create I-graph object
                ig_partition = ig.community_multilevel(weights='weight', resolution=resolution)  # Run Louvain partition
                ig_modularity = ig.modularity(ig_partition.membership, weights='weight')  # Modularity
                ig_time = time.time() - start_time  # Time taken

                # Append results for averaging later
                igraph_times.append(ig_time)
                igraph_modularities.append(ig_modularity)

                # Free up memory
                del ig, ig_partition, adj
                gc.collect()

            row_result['igraph_time'] = np.mean(igraph_times)
            row_result['igraph_modularity'] = np.mean(igraph_modularities)

        # # Check if the number of rows exceeds 15,000 for cuGraph
        # if sub_node_vectors.shape[0] > 15000:
        #     # Skip cuGraph if rows exceed threshold
        #     row_result['cugraph_time'] = 0
        #     row_result['cugraph_modularity'] = 0
        # else:
        #     # ---------------------- cuGraph Louvain Part ----------------------
        #     cugraph_times = []
        #     cugraph_modularities = []
        #     for _ in range(num_iterations):
        #         start_time = time.time()

        #         # Transfer PyTorch tensor to CuPy for cuGraph Louvain
        #         sub_node_vectors_np = sub_node_vectors.cpu().numpy()  # Convert PyTorch tensor to NumPy
        #         sub_node_vectors_cp = cp.asarray(sub_node_vectors_np)  # Convert NumPy to CuPy

        #         # Step 1: Compute cosine similarity using CuPy
        #         norms = cp.linalg.norm(sub_node_vectors_cp, axis=1)
        #         cosine_similarity_matrix = cp.dot(sub_node_vectors_cp, sub_node_vectors_cp.T) / (norms[:, None] * norms[None, :])

        #         # Step 2: Normalize cosine similarity between 0 and 1
        #         adj = (1 + cosine_similarity_matrix) / 2

        #         # Step 3: Fill diagonal with zeros (self-loop weights to 0)
        #         cp.fill_diagonal(adj, 0)

        #         # Step 4: Convert adjacency matrix to cuGraph input format (source, destination, weight)
        #         src, dst = cp.nonzero(adj)
        #         weights = adj[src, dst]

        #         # Step 5: Convert CuPy arrays to cuDF DataFrame
        #         adj_df = cudf.DataFrame({
        #             'src': src.get(),
        #             'dst': dst.get(),
        #             'weight': weights.get()
        #         })

        #         # Remove self-loops (if needed)
        #         adj_df = adj_df[adj_df['src'] != adj_df['dst']]

        #         # Step 6: Create cuGraph Graph object and add edges
        #         G = cugraph.Graph(directed=False)
        #         G.from_cudf_edgelist(adj_df, source='src', destination='dst', edge_attr='weight')

        #         # Step 7: Run the Louvain algorithm with a resolution parameter
        #         louvain_results, modularity_score = cugraph.louvain(G, resolution=resolution)
        #         cu_time = time.time() - start_time

        #         # Append results for averaging later
        #         cugraph_times.append(cu_time)
        #         cugraph_modularities.append(modularity_score)

        #         # Free up memory
        #         del sub_node_vectors_cp, adj, adj_df, G
        #         cp.get_default_memory_pool().free_all_blocks()

        #     # Record cuGraph Louvain results
        #     row_result['cugraph_time'] = np.mean(cugraph_times)
        #     row_result['cugraph_modularity'] = np.mean(cugraph_modularities)

        # # ---------------------- Graph-less Louvain Part ----------------------
        # graphless_times = []
        # graphless_modularities = []
        # for _ in range(num_iterations):
        #     start_time = time.time()

        #     # Normalize node vectors for graphless Louvain
        #     norms = np.linalg.norm(sub_node_vectors, axis=1, keepdims=True)
        #     sub_node_vectors_norm = sub_node_vectors / norms
        #     sub_node_vectors_norm = torch.cat((sub_node_vectors_norm, torch.ones(sub_node_vectors.size(0), 1)), dim=1)
        #     sub_node_vectors_norm = sub_node_vectors_norm / torch.sqrt(torch.tensor(2.0))

        #     # Run graphless Louvain
        #     my_partitions = louvain_partition_gpu(sub_node_vectors_norm.to('cuda'), gamma=resolution, threshold=1e-7, max_level=-1, seed=42)
        #     final_communities = get_final_communities(my_partitions)
        #     graphless_modularity = modularity_all_partitions(sub_node_vectors_norm, final_communities.to('cpu'))
        #     graphless_time = time.time() - start_time  # Time taken

        #     # Append results for averaging later
        #     graphless_times.append(graphless_time)
        #     graphless_modularities.append(graphless_modularity)

        # # Record Graph-less Louvain results
        # row_result['graphless_time'] = np.mean(graphless_times)
        # row_result['graphless_modularity'] = np.mean(graphless_modularities)

        # Append row to results
        results.append(row_result)

    # Convert results to DataFrame
    results_df = pd.DataFrame(results)
    return results_df


In [ ]:
l = [(i+1)/20 for i in range(20)]
l.sort()
l

[0.05,
 0.1,
 0.15,
 0.2,
 0.25,
 0.3,
 0.35,
 0.4,
 0.45,
 0.5,
 0.55,
 0.6,
 0.65,
 0.7,
 0.75,
 0.8,
 0.85,
 0.9,
 0.95,
 1.0]

In [ ]:
run_experiments(Flickr_data.data.x[~is_row_zero], l, 5)

/usr/local/lib/python3.10/dist-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


,threshold,num_nodes,igraph_time,igraph_modularity
0,0.05,4462,14.265100,0.016855
1,0.10,8924,67.722143,0.016782
2,0.15,13387,144.846427,0.016567
3,0.20,17849,405.932551,0.016660
4,0.25,22312,611.608445,0.016661
5,0.30,26774,0.000000,0.000000
6,0.35,31237,0.000000,0.000000
7,0.40,35699,0.000000,0.000000
8,0.45,40162,0.000000,0.000000
9,0.50,44624,0.000000,0.000000


In [ ]:
cp.get_default_memory_pool().free_all_blocks()

## AmazonProducts

In [ ]:
from torch_geometric.datasets import AmazonProducts

In [ ]:
AmazonProducts_data = AmazonProducts(root='data/AmazonProducts')

Processing...
Done!


In [ ]:
AmazonProducts_data.data

/usr/local/lib/python3.10/dist-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


Data(x=[1569960, 200], edge_index=[2, 264339468], y=[1569960, 107], train_mask=[1569960], val_mask=[1569960], test_mask=[1569960])

In [ ]:
AmazonProducts_data.data['x']

tensor([[-0.1466,  0.2226, -0.3597,  ...,  0.1699,  0.8974,  1.6527],
        [-0.2805,  0.0190,  0.4301,  ..., -1.1758, -1.8365, -1.1693],
        [ 0.2554,  0.2519, -0.0291,  ...,  1.3751, -0.0735,  0.6262],
        ...,
        [-0.8121,  0.3626, -0.7781,  ...,  0.0639,  0.8645,  0.0389],
        [ 1.5977, -2.3989, -0.0569,  ..., -1.4413,  0.2966,  0.0985],
        [-0.1663,  0.0629, -0.0474,  ...,  0.1853, -0.1216, -0.9181]])

In [ ]:
Amazon_is_row_zero = (AmazonProducts_data.data.x == 0).all(dim=1)
AmazonProducts_data[Amazon_is_row_zero]

AmazonProducts()

In [ ]:
l = [0.002, 0.004, 0.006, 0.008, 0.01]

In [ ]:
run_experiments(AmazonProducts_data.data.x, l, 5)

,threshold,num_nodes,igraph_time,igraph_modularity
0,0.002,3139,18.263729,0.015434
1,0.004,6279,121.145025,0.014941
2,0.006,9419,389.508216,0.014661
3,0.008,12559,859.967906,0.014587
4,0.010,15699,1598.963875,0.014510


In [ ]:
run_experiments(AmazonProducts_data.data.x, [0.5, 1], 1)

/usr/local/lib/python3.10/dist-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


,threshold,num_nodes,igraph_time,igraph_modularity
0,0.5,784980,0,0
1,1.0,1569960,0,0


# Yelp

In [ ]:
from torch_geometric.datasets import Yelp

In [ ]:
Yelp_data = Yelp(root='data/Yelp')

Processing...
Done!


In [ ]:
Yelp_data.data.x.shape

/usr/local/lib/python3.10/dist-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


torch.Size([716847, 300])

In [ ]:
Yelp_is_row_zero = (Yelp_data.data.x == 0).all(dim=1)
Yelp_data.data.x[Yelp_is_row_zero].shape

torch.Size([70, 300])

In [ ]:
l = [0.004, 0.008, 0.012, 0.016, 0.02]

In [ ]:
run_experiments(Yelp_data.data.x[~Yelp_is_row_zero], l, 5)

,threshold,num_nodes,igraph_time,igraph_modularity
0,0.004,2867,2.958928,0.003569
1,0.008,5734,13.912348,0.003675
2,0.012,8601,38.568178,0.003770
3,0.016,11468,78.979557,0.003789
4,0.020,14335,122.356389,0.003813


In [ ]:
run_experiments(Yelp_data.data.x[~Yelp_is_row_zero], [0.5, 1], 1)

/usr/local/lib/python3.10/dist-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


,threshold,num_nodes,igraph_time,igraph_modularity
0,0.5,358388,0,0
1,1.0,716777,0,0


# Taobao

In [ ]:
from torch_geometric.datasets import Taobao

In [ ]:
Taobao_data = Taobao(root='data/Taobao')

In [ ]:
u_t_i = Taobao_data.data['user', 'to', 'item']['edge_index']
i_t_c = Taobao_data.data['item', 'to', 'category']['edge_index']

/usr/local/lib/python3.10/dist-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


In [ ]:
def join_relationships_count(type_a_to_intermediate_tensor, intermediate_to_type_b_tensor, type_a_name='type_A', type_b_name='type_B'):
    """
    Joins two torch tensors using an intermediate entity as the intermediary and counts the occurrences of each pair.

    Parameters:
    - type_a_to_intermediate_tensor (torch.Tensor): A tensor with shape [2, n] representing relationships between type A and intermediate.
    - intermediate_to_type_b_tensor (torch.Tensor): A tensor with shape [2, m] representing relationships between intermediate and type B.
    - type_a_name (str): Name of the first type (e.g., 'type_A').
    - type_b_name (str): Name of the second type (e.g., 'type_B').

    Returns:
    - pd.DataFrame: A DataFrame with columns 'type_A', 'type_B', and 'count'.
    """
    # Extract rows for easier understanding
    type_a_ids = type_a_to_intermediate_tensor[0]  # IDs for type A (e.g., authors)
    intermediate_ids_from_type_a = type_a_to_intermediate_tensor[1]  # intermediate IDs related to type A

    intermediate_ids_from_type_b = intermediate_to_type_b_tensor[0]  # intermediate IDs related to type B
    type_b_ids = intermediate_to_type_b_tensor[1]  # IDs for type B (e.g., venues)

    # Convert tensors to lists for easier manipulation
    type_a_ids = type_a_ids.tolist()
    intermediate_ids_from_type_a = intermediate_ids_from_type_a.tolist()
    intermediate_ids_from_type_b = intermediate_ids_from_type_b.tolist()
    type_b_ids = type_b_ids.tolist()

    # Create a dictionary that maps intermediate IDs to type B
    intermediate_to_type_b = {}
    for intermediate_id, type_b_id in zip(intermediate_ids_from_type_b, type_b_ids):
        if intermediate_id not in intermediate_to_type_b:
            intermediate_to_type_b[intermediate_id] = []
        intermediate_to_type_b[intermediate_id].append(type_b_id)

    # Create the joined data for type A and type B via intermediate
    joined_data = []
    for type_a_id, intermediate_id in zip(type_a_ids, intermediate_ids_from_type_a):
        if intermediate_id in intermediate_to_type_b:
            for type_b_id in intermediate_to_type_b[intermediate_id]:
                joined_data.append((type_a_id, type_b_id))

    # Convert to a DataFrame and count occurrences
    df = pd.DataFrame(joined_data, columns=[type_a_name, type_b_name])
    count_df = df.groupby([type_a_name, type_b_name]).size().reset_index(name='count')

    return count_df

taobao_df = join_relationships_count(u_t_i, i_t_c)

In [ ]:
def filter_by_threshold(df, threshold_percentage=1, column_to_filter=1):
    """
    Filter the DataFrame based on a count threshold applied to either the first or second column.

    Parameters:
    - df (pd.DataFrame): The input DataFrame with at least two columns.
    - threshold_percentage (float): The threshold percentage of the maximum count.
    - filter_on (str): Specify 'first' to filter based on the first column, or 'second' to filter based on the second column. Default is 'second'.

    Returns:
    - pd.DataFrame: A filtered DataFrame where entries in the specified column meet the count threshold.
    """

    # Calculate the total number of unique values in the selected column
    col_counts = df.iloc[:, column_to_filter].value_counts()

    # Calculate the maximum count (the most frequent value in the selected column)
    max_count = col_counts.max()
    print(max_count)
    # Calculate the threshold count
    threshold = max_count * (threshold_percentage / 100)

    # Filter the DataFrame based on the threshold
    filtered_values = col_counts[col_counts >= threshold].index
    filtered_df = df[df.iloc[:, column_to_filter].isin(filtered_values)]

    return filtered_df

In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
def df_to_matrix(df):
    [c1, c2, c3] = df.columns.tolist()
    pivot_df = df.pivot(index=c1, columns=c2, values=c3)

    pivot_df.fillna(0, inplace=True)

    return pivot_df
def convert_to_tfidf(pivot_df):

    # Step 2: Apply TF-IDF transformation
    tfidf_transformer = TfidfTransformer()
    tfidf_matrix = tfidf_transformer.fit_transform(pivot_df)

    # Step 3: Convert the resulting matrix back to a DataFrame for easier analysis
    tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), index=pivot_df.index, columns=pivot_df.columns)

    return tfidf_df

In [ ]:
filtered_taobao_df = filter_by_threshold(taobao_df, 15)

417880


In [ ]:
filtered_taobao_df.type_B.nunique()

66

In [ ]:
pivot_df = df_to_matrix(filtered_taobao_df)
tfidf_df = convert_to_tfidf(pivot_df)
tfidf_matrix = torch.tensor(tfidf_df.values)

In [ ]:
tfidf_matrix

tensor([[0.4061, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.3960, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.8664, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],
       dtype=torch.float64)

In [ ]:
del pivot_df, tfidf_df, taobao_df, filtered_taobao_df
gc.collect()

0

In [ ]:
print(tfidf_matrix.shape)

torch.Size([936946, 66])


In [ ]:
l = [0.005, 0.01, 0.015, 0.02, 0.025, 0.03]

In [ ]:
print(torch.unique(u_t_i[0]).shape)
print(torch.unique(u_t_i[1]).shape)
print(torch.unique(i_t_c[0]).shape)
print(torch.unique(i_t_c[1]).shape)

torch.Size([987991])
torch.Size([4161138])
torch.Size([4161138])
torch.Size([9437])


In [ ]:
run_experiments(tfidf_matrix, l, 5)

,threshold,num_nodes,igraph_time,igraph_modularity
0,0.005,4684,22.442026,0.020711
1,0.010,9369,85.716797,0.021091
2,0.015,14054,269.230929,0.021321
3,0.020,18738,0.000000,0.000000
4,0.025,23423,0.000000,0.000000
5,0.030,28108,0.000000,0.000000


In [ ]:
gc.collect()

In [ ]:
run_experiments(tfidf_matrix, [0.5, 1], 5)